# M3L2 E10 - Refactorizar el Caos (Resolution)

Una posible solucion al ejercicio de refactorizacion.
Hay multiples formas validas de resolver esto.


In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")


## Problemas identificados en el script original

1. Se crea un nuevo `OpenAI()` en cada llamada (deberia crearse una vez).
2. El prompt es un string manual con concatenacion.
3. La logica de idioma esta mezclada con la construccion del prompt.
4. Se manda todo el contexto (PRODUCT_DATA + FAQ_DATA) sin retrieval real.
5. El modelo y la temperatura estan hardcodeados en la funcion.
6. No hay forma de debuggear que parte del contexto se uso.
7. Ingestion y consulta estan mezcladas en una sola funcion.


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# =====================================================================
# CONFIGURACION CENTRALIZADA
# Un solo lugar para cambiar modelo, temperatura, k, etc.
# =====================================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # temperatura 0 para consistencia
embeddings = OpenAIEmbeddings()
parser = StrOutputParser()

print("Configuracion centralizada:")
print(f"  LLM: {llm.model_name}, temperature={llm.temperature}")


In [ ]:
# =====================================================================
# INGESTION - se hace una vez
# Documentos separados por tema para que el retrieval sea preciso
# =====================================================================

DOCUMENTOS = [
    "Plan Basico: $9.99 por mes. Hasta 5 usuarios. 10GB almacenamiento. Soporte por email.",
    "Plan Pro: $29.99 por mes. Hasta 20 usuarios. 100GB almacenamiento. Soporte email y chat.",
    "Plan Enterprise: $99.99 por mes. Usuarios ilimitados. 1TB almacenamiento. Soporte completo con SLA 24/7.",
    "Cancelacion: se puede cancelar en cualquier momento desde el panel. Sin penalidades.",
    "Prueba gratuita: 14 dias gratis sin tarjeta de credito.",
    "Cambio de plan: upgrade o downgrade en cualquier momento. Aplica desde el proximo ciclo.",
]

vectorstore = FAISS.from_texts(DOCUMENTOS, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Vector store creado con {len(DOCUMENTOS)} documentos.")

# Verificar que el retriever funciona
docs_test = retriever.invoke("Plan Pro precio")
print(f"Test retriever 'Plan Pro precio': {len(docs_test)} docs")
for doc in docs_test:
    print(f"  - {doc.page_content[:60]}...")


In [ ]:
# =====================================================================
# PROMPT - separado de la logica de idioma
# El idioma se puede manejar en el system message si es necesario
# o como una variable del template
# =====================================================================

support_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente de soporte de producto. "
        "Responde usando solo la informacion del contexto. "
        "Si la respuesta no esta en el contexto, indica que no tienes esa informacion."
    ),
    (
        "human",
        "Contexto:\n{context}\n\nPregunta:\n{question}"
    )
])

print(f"Prompt variables: {support_prompt.input_variables}")


In [ ]:
# =====================================================================
# CHAIN DE CONSULTA - flujo visible con LCEL
# =====================================================================

def format_docs(docs) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


support_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | support_prompt
    | llm
    | parser
)


def ask_chatbot_refactored(question: str) -> str:
    """
    Chatbot refactorizado con LangChain.
    - Modelo configurado en un solo lugar
    - Retrieval real: solo los documentos relevantes
    - Prompt estructurado como ChatPromptTemplate
    - Flujo visible con LCEL
    """
    return support_chain.invoke(question)


print("Chain de consulta creada.")
print(f"Tipo: {type(support_chain).__name__}")


In [ ]:
# Probar el chatbot refactorizado
print("=== Chatbot refactorizado ===")
print(ask_chatbot_refactored("Cuanto cuesta el plan Pro?"))
print()
print(ask_chatbot_refactored("Hay prueba gratuita?"))


## Debugging modular: la ventaja clave


In [ ]:
question = "Cuanto cuesta el plan Pro?"

print("=== Inspeccion del pipeline ===")

print("[Retrieval]")
docs = retriever.invoke(question)
for doc in docs:
    print(f"  - {doc.page_content}")
print()

print("[Contexto formateado]")
context = format_docs(docs)
print(context)
print()

print("[Respuesta final]")
print(support_chain.invoke(question))


## Reflexion: respuestas esperadas

1. **Componentes independientes**: 4 (LLM, Embeddings, Retriever, Prompt)
2. **Para cambiar el modelo**: cambiar 1 linea (`ChatOpenAI(model="...")`)
3. **Para agregar documentos**: agregar items a `DOCUMENTOS` y recrear el vector store
4. **Para debuggear**: `retriever.invoke(q)` muestra los docs; `format_docs(docs)` muestra el contexto
5. **Chain vs Agent**: alcanza con Chain porque el flujo es fijo (siempre buscar y responder)

## Checklist


In [ ]:
checklist = {
    "Separaste ingestion y consulta": True,
    "El retriever esta encapsulado": True,
    "El prompt usa ChatPromptTemplate": True,
    "El modelo esta configurado centralmente": True,
    "El flujo esta definido como Chain o LCEL": True,
    "No hay API keys hardcodeadas": True,
    "El modelo es configurable (no hardcodeado)": True,
    "Se pueden inspeccionar los documentos recuperados": True,
    "Se puede ver el prompt final": True,
}

print("=== Checklist de produccion ===")
aprobados = 0
for item, estado in checklist.items():
    simbolo = "[OK]" if estado else "[NO]"
    if estado:
        aprobados += 1
    print(f"  {simbolo} {item}")

print(f"\nAprobados: {aprobados}/{len(checklist)}")
print("M3L2 E10 Resolution completado.")
